# EMG-ASL Model Comparison: LSTM vs CNN-LSTM vs SVM

This notebook benchmarks the three model families available in this project on
a shared synthetic sEMG dataset so their validation accuracies, per-class
performance, and inference latencies can be compared side by side.

**Models under comparison**

| Model | Input | Feature extraction |
|---|---|---|
| `ASLEMGClassifier` (LSTM) | 80-dim hand-crafted feature vector | `src.utils.features.extract_features` |
| `CNNLSTMClassifier` (CNN-LSTM) | Raw window (40 samples × 8 channels) | Learned by the CNN front-end |
| `SVC` (SVM) | 80-dim hand-crafted feature vector | `src.utils.features.extract_features` |

**Notebook structure**
1. Setup — imports, constants, reproducibility seed
2. Synthetic data generation
3. Dataset loading and windowing
4. Feature extraction for SVM
5. Stratified 80/20 train/val split
6. LSTM training
7. CNN-LSTM training
8. SVM training
9. Results comparison table + per-class accuracy bar chart
10. Confusion matrices (side by side)
11. Inference latency benchmark

## 1 · Setup

In [ ]:
# ─── Imports & constants ─────────────────────────────────────────────────────
import sys
import os
import time
import warnings
from pathlib import Path

REPO_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

from src.data.synthetic import generate_dataset
from src.data.loader import load_dataset, create_windows
from src.utils.features import extract_features, get_feature_names
from src.utils.constants import (
    ASL_LABELS,
    N_CHANNELS,
    NUM_CLASSES,
    SAMPLE_RATE,
    WINDOW_SIZE_MS,
    FEATURE_VECTOR_SIZE,
)
from src.models.lstm_classifier import ASLEMGClassifier
from src.models.cnn_lstm_classifier import CNNLSTMClassifier
from src.models.train import TrainConfig, train_model

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SYNTH_DIR = str(REPO_ROOT / 'data' / 'raw' / 'synthetic_comparison')
MODEL_DIR  = str(REPO_ROOT / 'models' / 'comparison')

# Subset of labels to keep training fast in the notebook
# Use all 36 classes — reduce N_REPS instead for speed
N_PARTICIPANTS = 3   # participants used for synthetic generation
N_REPS         = 10  # repetitions per label per participant
EPOCHS         = 30  # fast enough for a demo notebook

print(f'Repository root : {REPO_ROOT}')
print(f'Device          : {DEVICE}')
print(f'Torch version   : {torch.__version__}')
print(f'Num classes     : {NUM_CLASSES}')
print(f'Feature size    : {FEATURE_VECTOR_SIZE}')

## 2 · Synthetic Data Generation

`generate_dataset` writes one CSV per participant to `SYNTH_DIR`.  If the files
already exist from a previous run the generation step is skipped.

In [ ]:
# ─── Generate synthetic sEMG dataset ────────────────────────────────────────
synth_path = Path(SYNTH_DIR)
existing = list(synth_path.glob('*.csv')) if synth_path.exists() else []

if len(existing) >= N_PARTICIPANTS:
    print(f'Found {len(existing)} existing CSVs in {synth_path}. Skipping generation.')
else:
    print(f'Generating {N_PARTICIPANTS} participants × {N_REPS} reps × {NUM_CLASSES} labels …')
    generate_dataset(
        output_dir=SYNTH_DIR,
        n_participants=N_PARTICIPANTS,
        n_reps=N_REPS,
        labels=list(ASL_LABELS),
    )

print('Done.')

## 3 · Load Dataset and Create Windows

`create_windows` produces:
- `windows` — shape `(N, 40, 8)` — raw time-series windows
- `labels`  — shape `(N,)`      — string class labels

The LSTM and SVM receive *flattened / feature-extracted* versions of these
windows; the CNN-LSTM receives the raw `(N, 40, 8)` tensor directly.

In [ ]:
# ─── Load and window the dataset ─────────────────────────────────────────────
df = load_dataset(SYNTH_DIR)

windows, labels = create_windows(
    df,
    window_size_ms=WINDOW_SIZE_MS,
    overlap=0.5,
    fs=SAMPLE_RATE,
)

print(f'\nWindows shape : {windows.shape}   (N, T, C)')
print(f'Labels shape  : {labels.shape}')
print(f'Unique labels : {len(np.unique(labels))}')

# Quick sanity check on class balance
unique, counts = np.unique(labels, return_counts=True)
print(f'Min windows per class : {counts.min()}')
print(f'Max windows per class : {counts.max()}')
print(f'Mean windows per class: {counts.mean():.1f}')

## 4 · Feature Extraction for SVM (and LSTM)

`extract_features` returns an 80-dim vector per window
(5 time-domain + 5 frequency-domain features × 8 channels).
These are used by both the LSTM and the SVM.

In [ ]:
# ─── Extract hand-crafted feature vectors ────────────────────────────────────
print('Extracting features …  (may take ~30 s for large N)')
t0 = time.time()

feature_matrix = np.stack(
    [extract_features(windows[i], fs=float(SAMPLE_RATE)) for i in range(len(windows))],
    axis=0,
)  # (N, 80)

elapsed = time.time() - t0
feature_names = get_feature_names(N_CHANNELS)

print(f'Feature matrix shape : {feature_matrix.shape}')
print(f'Extraction time      : {elapsed:.1f} s  ({elapsed/len(windows)*1000:.2f} ms/window)')
print(f'Feature vector size  : {feature_matrix.shape[1]}  (expected {FEATURE_VECTOR_SIZE})')

# Summary stats
feat_df = pd.DataFrame(feature_matrix, columns=feature_names)
print('\nFeature stats (first 6 features):')
print(feat_df.iloc[:, :6].describe().round(4))

## 5 · Stratified 80/20 Train / Val Split

A per-class stratified split keeps class proportions identical in both halves.
All three models train on the same split for a fair comparison.

In [ ]:
# ─── Stratified 80 / 20 split ────────────────────────────────────────────────
VAL_FRAC = 0.20
rng = np.random.default_rng(SEED)

train_idx, val_idx = [], []
for lbl in np.unique(labels):
    idxs = np.where(labels == lbl)[0]
    rng.shuffle(idxs)
    n_val = max(1, int(len(idxs) * VAL_FRAC))
    val_idx.extend(idxs[:n_val].tolist())
    train_idx.extend(idxs[n_val:].tolist())

train_idx = np.array(train_idx)
val_idx   = np.array(val_idx)
rng.shuffle(train_idx)
rng.shuffle(val_idx)

# Raw windows (CNN-LSTM)
raw_train_w, raw_val_w = windows[train_idx], windows[val_idx]
train_lbl,   val_lbl   = labels[train_idx],  labels[val_idx]

# Feature vectors (LSTM + SVM)
feat_train, feat_val = feature_matrix[train_idx], feature_matrix[val_idx]

print(f'Train windows : {len(raw_train_w):,}')
print(f'Val windows   : {len(raw_val_w):,}')
print(f'Train labels  : {train_lbl.shape}')
print(f'Val labels    : {val_lbl.shape}')

# Integer label encoding (shared across all three models)
le = LabelEncoder()
le.fit(list(ASL_LABELS))
train_y = le.transform(train_lbl)
val_y   = le.transform(val_lbl)

## 6 · LSTM Training

`ASLEMGClassifier` ingests the 80-dim hand-crafted feature vector.
`train_model` handles the full training loop with early stopping.

In [ ]:
# ─── Train ASLEMGClassifier (LSTM) ───────────────────────────────────────────
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

lstm_cfg = TrainConfig(
    input_size   = FEATURE_VECTOR_SIZE,
    hidden_size  = 128,
    num_layers   = 2,
    num_classes  = NUM_CLASSES,
    dropout      = 0.3,
    lr           = 1e-3,
    weight_decay = 1e-4,
    batch_size   = 64,
    epochs       = EPOCHS,
    early_stopping_patience = 8,
    checkpoint_dir  = MODEL_DIR,
    checkpoint_name = 'best_lstm.pt',
    device          = DEVICE,
    wandb_project   = '',    # disabled
    label_names     = list(ASL_LABELS),
)

print(f'Training LSTM  |  epochs={EPOCHS}  device={DEVICE}')
print(f'Input size     : {lstm_cfg.input_size}')
print()

t_lstm_start = time.time()
lstm_model = train_model(
    train_windows = feat_train,
    train_labels  = train_lbl,
    val_windows   = feat_val,
    val_labels    = val_lbl,
    config        = lstm_cfg,
)
lstm_train_time = time.time() - t_lstm_start

# Validation accuracy
lstm_model.eval()
with torch.no_grad():
    X_val_t = torch.from_numpy(feat_val.astype(np.float32))
    lstm_logits = lstm_model(X_val_t)
    lstm_preds  = lstm_logits.argmax(dim=1).numpy()

lstm_val_acc = accuracy_score(val_y, lstm_preds)
print(f'\nLSTM val accuracy : {lstm_val_acc:.4f}  ({lstm_val_acc:.1%})')
print(f'Training time     : {lstm_train_time:.1f} s')

## 7 · CNN-LSTM Training

`CNNLSTMClassifier` operates directly on raw windows of shape `(batch, 40, 8)`.
No hand-crafted feature extraction is needed — the CNN learns spatial filters.

In [ ]:
# ─── Train CNNLSTMClassifier ─────────────────────────────────────────────────
cnn_lstm_model = CNNLSTMClassifier(
    num_classes  = NUM_CLASSES,
    dropout      = 0.3,
    label_names  = list(ASL_LABELS),
).to(DEVICE)

criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = torch.optim.Adam(cnn_lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cnn, mode='min', patience=4, factor=0.5
)

# Build DataLoaders for raw windows
def make_raw_loader(raw_w, y_int, batch_size=64, shuffle=True):
    X = torch.from_numpy(raw_w.astype(np.float32))
    y = torch.from_numpy(y_int.astype(np.int64))
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

cnn_train_loader = make_raw_loader(raw_train_w, train_y, shuffle=True)
cnn_val_loader   = make_raw_loader(raw_val_w,   val_y,   shuffle=False)

print(f'Training CNN-LSTM  |  epochs={EPOCHS}  device={DEVICE}')
print(f'Raw window shape   : {raw_train_w.shape[1:]}  (T, C)')
print()

best_cnn_val_loss = float('inf')
no_improve_cnn    = 0
EARLY_STOP        = 8

t_cnn_start = time.time()
device_t    = torch.device(DEVICE)

for epoch in range(1, EPOCHS + 1):
    # Train
    cnn_lstm_model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for Xb, yb in cnn_train_loader:
        Xb, yb = Xb.to(device_t), yb.to(device_t)
        optimizer_cnn.zero_grad()
        logits = cnn_lstm_model(Xb)
        loss   = criterion_cnn(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(cnn_lstm_model.parameters(), 1.0)
        optimizer_cnn.step()
        tr_loss    += loss.item() * len(yb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total   += len(yb)

    # Validate
    cnn_lstm_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in cnn_val_loader:
            Xb, yb = Xb.to(device_t), yb.to(device_t)
            logits  = cnn_lstm_model(Xb)
            loss    = criterion_cnn(logits, yb)
            val_loss    += loss.item() * len(yb)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total   += len(yb)

    ep_tr_loss  = tr_loss  / tr_total
    ep_val_loss = val_loss / val_total
    ep_tr_acc   = tr_correct  / tr_total
    ep_val_acc  = val_correct / val_total
    scheduler_cnn.step(ep_val_loss)

    print(
        f'Epoch {epoch:3d}/{EPOCHS}  '
        f'train_loss={ep_tr_loss:.4f}  train_acc={ep_tr_acc:.3f}  '
        f'val_loss={ep_val_loss:.4f}  val_acc={ep_val_acc:.3f}'
    )

    if ep_val_loss < best_cnn_val_loss:
        best_cnn_val_loss = ep_val_loss
        no_improve_cnn    = 0
        cnn_lstm_model.save(Path(MODEL_DIR) / 'best_cnn_lstm.pt')
    else:
        no_improve_cnn += 1
        if no_improve_cnn >= EARLY_STOP:
            print(f'[early stopping] No improvement for {no_improve_cnn} epochs.')
            break

cnn_train_time = time.time() - t_cnn_start

# Reload best checkpoint and evaluate
cnn_lstm_model = CNNLSTMClassifier.load(Path(MODEL_DIR) / 'best_cnn_lstm.pt')
cnn_lstm_model.eval()

with torch.no_grad():
    X_raw_val_t  = torch.from_numpy(raw_val_w.astype(np.float32))
    cnn_logits   = cnn_lstm_model(X_raw_val_t)
    cnn_preds    = cnn_logits.argmax(dim=1).numpy()

cnn_val_acc = accuracy_score(val_y, cnn_preds)
print(f'\nCNN-LSTM val accuracy : {cnn_val_acc:.4f}  ({cnn_val_acc:.1%})')
print(f'Training time         : {cnn_train_time:.1f} s')

## 8 · SVM Training

A `StandardScaler → SVC(RBF kernel)` pipeline is trained on the same 80-dim
feature vectors used by the LSTM.  `joblib` is used for serialisation so the
fitted pipeline can be reloaded for deployment.

In [ ]:
# ─── Train SVM (StandardScaler + RBF SVC) ───────────────────────────────────
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(kernel='rbf', C=10.0, gamma='scale',
                   decision_function_shape='ovr',
                   random_state=SEED)),
])

print('Training SVM  |  kernel=RBF  C=10  gamma=scale')
print(f'Train samples : {len(feat_train):,}')
print()

t_svm_start = time.time()
svm_pipeline.fit(feat_train, train_lbl)
svm_train_time = time.time() - t_svm_start

svm_preds_str = svm_pipeline.predict(feat_val)
svm_preds     = le.transform(svm_preds_str)
svm_val_acc   = accuracy_score(val_y, svm_preds)

# Persist pipeline
svm_save_path = Path(MODEL_DIR) / 'svm_pipeline.joblib'
joblib.dump(svm_pipeline, svm_save_path)

print(f'SVM val accuracy  : {svm_val_acc:.4f}  ({svm_val_acc:.1%})')
print(f'Training time     : {svm_train_time:.1f} s')
print(f'Pipeline saved to : {svm_save_path}')

## 9 · Results Comparison

Summary table of overall val accuracy followed by a per-class accuracy bar
chart for all three models.

In [ ]:
# ─── Results summary table ───────────────────────────────────────────────────
results = pd.DataFrame([
    {'Model': 'LSTM',     'Val Accuracy': lstm_val_acc, 'Train Time (s)': lstm_train_time},
    {'Model': 'CNN-LSTM', 'Val Accuracy': cnn_val_acc,  'Train Time (s)': cnn_train_time},
    {'Model': 'SVM',      'Val Accuracy': svm_val_acc,  'Train Time (s)': svm_train_time},
])
results['Val Accuracy (%)'] = (results['Val Accuracy'] * 100).round(2)
results['Train Time (s)']   = results['Train Time (s)'].round(1)
print('=== Validation Accuracy Summary ===')
print(results[['Model', 'Val Accuracy (%)', 'Train Time (s)']].to_string(index=False))

# ─── Per-class accuracy bar chart ────────────────────────────────────────────
def per_class_accuracy(y_true_int, y_pred_int, n_classes, label_names):
    """Return per-class accuracy as a dict {label: accuracy}."""
    accs = {}
    for cls_idx in range(n_classes):
        mask = y_true_int == cls_idx
        if mask.sum() == 0:
            continue
        accs[label_names[cls_idx]] = (y_pred_int[mask] == cls_idx).mean()
    return accs

label_names_list = list(ASL_LABELS)
pc_lstm    = per_class_accuracy(val_y, lstm_preds, NUM_CLASSES, label_names_list)
pc_cnn     = per_class_accuracy(val_y, cnn_preds,  NUM_CLASSES, label_names_list)
pc_svm     = per_class_accuracy(val_y, svm_preds,  NUM_CLASSES, label_names_list)

pc_df = pd.DataFrame({
    'Label':    list(pc_lstm.keys()),
    'LSTM':     list(pc_lstm.values()),
    'CNN-LSTM': [pc_cnn.get(k, 0.0) for k in pc_lstm.keys()],
    'SVM':      [pc_svm.get(k, 0.0) for k in pc_lstm.keys()],
})

x = np.arange(len(pc_df))
width = 0.28
colors_bar = ['steelblue', 'tomato', 'seagreen']

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(x - width, pc_df['LSTM'],     width, label='LSTM',     color=colors_bar[0], alpha=0.85)
ax.bar(x,         pc_df['CNN-LSTM'], width, label='CNN-LSTM', color=colors_bar[1], alpha=0.85)
ax.bar(x + width, pc_df['SVM'],      width, label='SVM',      color=colors_bar[2], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(pc_df['Label'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Per-class Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('Per-class Validation Accuracy — LSTM vs CNN-LSTM vs SVM')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Mean lines
for acc, col in zip([lstm_val_acc, cnn_val_acc, svm_val_acc], colors_bar):
    ax.axhline(acc, color=col, linestyle='--', lw=1.2, alpha=0.7)

plt.tight_layout()
plt.show()

## 10 · Confusion Matrices

Side-by-side normalised confusion matrices for all three models.

In [ ]:
# ─── Confusion matrices — side by side ───────────────────────────────────────
# Only include classes that appear in val_y (may be a subset when labels are sparse)
present_classes = sorted(set(val_y.tolist()))
present_names   = [label_names_list[i] for i in present_classes]

cm_lstm = confusion_matrix(val_y, lstm_preds, labels=present_classes, normalize='true')
cm_cnn  = confusion_matrix(val_y, cnn_preds,  labels=present_classes, normalize='true')
cm_svm  = confusion_matrix(val_y, svm_preds,  labels=present_classes, normalize='true')

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Normalised Confusion Matrices (row = true class)', fontsize=13)

for ax, cm, title, acc in zip(
    axes,
    [cm_lstm, cm_cnn, cm_svm],
    ['LSTM', 'CNN-LSTM', 'SVM'],
    [lstm_val_acc, cnn_val_acc, svm_val_acc],
):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=present_names,
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45,
              values_format='.2f')
    ax.set_title(f'{title}  (val acc {acc:.2%})', fontsize=11)
    ax.tick_params(labelsize=6)

plt.tight_layout()
plt.show()

## 11 · Inference Latency Benchmark

Each model is called 1,000 times with a single sample to measure per-sample
latency on CPU.  PyTorch models are in `eval()` mode with `torch.no_grad()`.
For CNN-LSTM the raw window shape `(1, 40, 8)` is used; for LSTM and SVM the
80-dim feature vector is used.

In [ ]:
# ─── Inference latency: 1000 single-sample predictions ──────────────────────
N_BENCH = 1000

# Prepare single-sample tensors on CPU for a fair comparison
sample_feat   = torch.from_numpy(feat_val[:1].astype(np.float32))   # (1, 80)
sample_raw    = torch.from_numpy(raw_val_w[:1].astype(np.float32))  # (1, 40, 8)
sample_feat_np = feat_val[:1]                                         # (1, 80) numpy

lstm_cpu    = lstm_model.cpu().eval()
cnn_cpu     = cnn_lstm_model.cpu().eval()

# --- LSTM ---
with torch.no_grad():
    for _ in range(20):   # warm-up
        _ = lstm_cpu(sample_feat)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_BENCH):
        _ = lstm_cpu(sample_feat)
lstm_ms = (time.perf_counter() - t0) * 1000 / N_BENCH

# --- CNN-LSTM ---
with torch.no_grad():
    for _ in range(20):
        _ = cnn_cpu(sample_raw)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_BENCH):
        _ = cnn_cpu(sample_raw)
cnn_ms = (time.perf_counter() - t0) * 1000 / N_BENCH

# --- SVM (includes feature scaling inside the pipeline) ---
for _ in range(20):
    _ = svm_pipeline.predict(sample_feat_np)
t0 = time.perf_counter()
for _ in range(N_BENCH):
    _ = svm_pipeline.predict(sample_feat_np)
svm_ms = (time.perf_counter() - t0) * 1000 / N_BENCH

# --- Report ---
latency_df = pd.DataFrame([
    {'Model': 'LSTM',     'ms / sample': round(lstm_ms, 4)},
    {'Model': 'CNN-LSTM', 'ms / sample': round(cnn_ms, 4)},
    {'Model': 'SVM',      'ms / sample': round(svm_ms, 4)},
])
print(f'=== Inference Latency ({N_BENCH} single-sample predictions, CPU) ===')
print(latency_df.to_string(index=False))
print()

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    latency_df['Model'],
    latency_df['ms / sample'],
    color=['steelblue', 'tomato', 'seagreen'],
    width=0.45,
    alpha=0.88,
)
for bar, val in zip(bars, latency_df['ms / sample']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{val:.3f} ms',
        ha='center', va='bottom', fontsize=9,
    )
ax.set_ylabel('ms / sample')
ax.set_title(f'Single-sample inference latency (CPU, n={N_BENCH})')
ax.set_ylim(0, latency_df['ms / sample'].max() * 1.3)
plt.tight_layout()
plt.show()

# Final combined summary
print('\n=== Final Comparison ===')
final_df = results[['Model', 'Val Accuracy (%)', 'Train Time (s)']].copy()
final_df['Latency (ms/sample)'] = latency_df['ms / sample'].values
print(final_df.to_string(index=False))